# Phase 25 — Reproducibility package (Study 2)
**Resolves:** R2-15 (feature lists + variable names, outcome construction, exclusions, seeds/split identifiers, preprocessing,
Optuna search spaces, signature code, software versions, hyper-parameters, executable code), R2-10 (signature definition),
R2 minor-6/7 inputs. Everything is extracted programmatically from the actual code/artefacts so the supplement cannot drift from the analysis.
**Outputs (reports/tables/revision/):** `S2_environment.csv`, `requirements_frozen.txt`, `S2_hyperparameters_all_models.csv`,
`S2_optuna_search_spaces.csv`, `S2_trial_budget.csv`, `S2_feature_inventory.csv`, `S2_outcome_definitions.csv`,
`S2_split_manifest_<dataset>.csv`, `S2_split_reproduction_check.csv`, `S2_code_manifest.csv`

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
import platform, subprocess, importlib.metadata as md, hashlib, glob
pk_ = ["numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn", "xgboost", "optuna", "shap", "torch", "matplotlib", "seaborn", "PyYAML", "joblib", "openpyxl"]
env_rows = [dict(item="python", version=platform.python_version()), dict(item="platform", version=platform.platform())]
for p in pk_:
    try: env_rows.append(dict(item=p, version=md.version(p)))
    except Exception: env_rows.append(dict(item=p, version="not installed"))
env_rows.append(dict(item="global_random_seed", version=str(SEED)))
save_table(pd.DataFrame(env_rows), ENV, "S2_environment.csv")
try:
    frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    open(os.path.join(ENV["REV"], "requirements_frozen.txt"), "w").write(frozen); print("requirements_frozen.txt written")
except Exception as e: print("pip freeze failed:", e)

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_environment.csv  (17 rows)
requirements_frozen.txt written


In [3]:
# Tuned hyper-parameters (every model, every dataset incl. the full-N sensitivity run if present)
hp = []
for tag in list(DATASETS) + ["arab5_depression_fullN_sensitivity"]:
    for md_ in ([DATASETS[tag]["models_dir"]] if tag in DATASETS else [tag]):
        f = os.path.join(ENV["MODELS"], md_, "best_hyperparameters.json")
        if not os.path.exists(f): continue
        for model, params in json.load(open(f)).items():
            hp.append(dict(dataset=tag, model=model, hyperparameters=json.dumps(params, sort_keys=True)))
save_table(pd.DataFrame(hp), ENV, "S2_hyperparameters_all_models.csv")

# Optuna search spaces, captured by calling the real suggest_params with a recording trial
import modeling
class RecTrial:
    def __init__(self): self.rec = []
    def suggest_int(self, n, lo, hi, **k): self.rec.append((n, "int", f"[{lo}, {hi}]")); return lo
    def suggest_float(self, n, lo, hi, log=False, **k): self.rec.append((n, "float" + (" (log)" if log else ""), f"[{lo}, {hi}]")); return lo
    def suggest_categorical(self, n, ch): self.rec.append((n, "categorical", str(list(ch)))); return ch[0]
ss = []
for m in MODEL_NAMES:
    t = RecTrial()
    try: modeling.suggest_params(t, m)
    except Exception as e: print("search-space capture failed for", m, e)
    ss += [dict(model=m, parameter=n, type=ty, range=r) for n, ty, r in t.rec]
save_table(pd.DataFrame(ss), ENV, "S2_optuna_search_spaces.csv")
cfg = ENV["CFG"]; maxtrain = max(len(load_split(t, ENV)["train"]) for t in DATASETS)
budget = [dict(model=m, optuna_trials=cfg["optuna"]["n_trials_tree"] if m in ("dt", "rf", "gb", "xgb") else cfg["optuna"]["n_trials_svm"],
               cv="5-fold StratifiedGroupKFold (group = feature signature), SMOTE inside each fold, objective = mean CV ROC-AUC",
               sampler="Optuna TPE, seed 42",
               training_subsample=(f"SVM subsampling cap {cfg['svm_subsample']['fit_rows']} rows > largest training set ({maxtrain}) => never triggered" if m == "svm" else "none")) for m in MODEL_NAMES]
save_table(pd.DataFrame(budget), ENV, "S2_trial_budget.csv"); pd.DataFrame(budget)

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_hyperparameters_all_models.csv  (35 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_optuna_search_spaces.csv  (26 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_trial_budget.csv  (7 rows)


,model,optuna_trials,cv,sampler,training_subsample
0,dt,30,5-fold StratifiedGroupKFold (group = feature s...,"Optuna TPE, seed 42",none
1,rf,30,5-fold StratifiedGroupKFold (group = feature s...,"Optuna TPE, seed 42",none
2,gb,30,5-fold StratifiedGroupKFold (group = feature s...,"Optuna TPE, seed 42",none
3,svm,15,5-fold StratifiedGroupKFold (group = feature s...,"Optuna TPE, seed 42",SVM subsampling cap 15000 rows > largest train...
4,nb,15,5-fold StratifiedGroupKFold (group = feature s...,"Optuna TPE, seed 42",none
5,knn,15,5-fold StratifiedGroupKFold (group = feature s...,"Optuna TPE, seed 42",none
6,xgb,30,5-fold StratifiedGroupKFold (group = feature s...,"Optuna TPE, seed 42",none


In [4]:
# Feature inventory + outcome definitions
inv = []
for tag, d in DATASETS.items():
    tr = load_split(tag, ENV)["train"]
    for c in features_of(tag):
        inv.append(dict(dataset=tag, variable=c, type="numeric" if c in d["numeric"] else "categorical",
                        n_levels_train=int(tr[c].nunique(dropna=True)) if c in d["categorical"] else np.nan,
                        pct_missing_train=round(100 * tr[c].isna().mean(), 2), readable_label="<< fill in for supplement >>"))
save_table(pd.DataFrame(inv), ENV, "S2_feature_inventory.csv")
outc = [
 dict(dataset="pakistan", outcome="Depression", construction="PHQ-9 sum score (column 'Scalling', sum of 9 item columns, match rate 1.0) >= 10 -> 'Depressed'", cutoff_type="standard clinical cutoff (Kroenke 2001)", excluded_predictors="9 PHQ-9 items; PHQ-9 sum score"),
 dict(dataset="arab5_depression", outcome="Depression", construction="PHQ-9 category (0-9 / 10-14 / 15+); positive = moderate or severe (>=10); complete-item rows only", cutoff_type="standard clinical cutoff", excluded_predictors="9 PHQ-9 items; PHQ-9 sum"),
 dict(dataset="arab5_anxiety", outcome="Anxiety", construction="GAD-7 category (0-4 / 5-9 / 10+); positive = mild or worse (>=5); complete-item rows only", cutoff_type="standard screening cutoff for mild anxiety (Spitzer 2006); NOT the >=10 'probable disorder' cutoff", excluded_predictors="7 GAD-7 items; GAD-7 sum"),
 dict(dataset="uganda", outcome="Anxiety", construction="continuous anxiety severity score (0-9) >= 4", cutoff_type="investigator-defined, data-driven (yields ~48-49% positives); no pre-registration or clinical validation available", excluded_predictors="none (no item-level columns among predictors)")]
save_table(pd.DataFrame(outc), ENV, "S2_outcome_definitions.csv")

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_feature_inventory.csv  (68 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_outcome_definitions.csv  (4 rows)


'C:\\maternal-mental-health-risk\\reports/tables\\revision\\S2_outcome_definitions.csv'

In [5]:
# Split manifest (hashed signatures, never raw data) + regeneration check from the interim files
from preprocessing import compute_group_id, three_way_group_split
for tag in DATASETS:
    S = load_split(tag, ENV); feats = features_of(tag); rows = []
    for k, nm in [("train", "training"), ("cal", "calibration"), ("test", "test")]:
        sig = compute_group_id(S[k][feats], feats)
        rows.append(pd.DataFrame(dict(dataset=tag, partition=nm, position=np.arange(len(sig)),
                                      signature_md5=[hashlib.md5(s.encode()).hexdigest()[:12] for s in sig], label=S[k][DATASETS[tag]["label"]].values)))
    pd.concat(rows).to_csv(os.path.join(ENV["REV"], f"S2_split_manifest_{tag}.csv"), index=False)
print("split manifests written")
SD_PK = ["Age", "Gestational Age", "Number of sons ", "Number of daughters", "Total Number of Children", "Gravida", "Female Education", "Husband Education",
         "Working Status", "Physical Health ", "Previous Miscarriage", "Sufficient Money for Basic Needs", "Current Appereance Acceptance",
         "Family System", "Male Gender Preference", "Relationship with Mother in-law"]
SD_AR = ["part1_country", "Locality_first", "WomenAge", "MariageAge", "NumberofPregnancy", "NumberofAbortions", "EducationLevel", "Work", "CovidDiagL1",
         "CovidDiagL2", "Health_Prob", "StressduringCovid", "FamilyProblems", "FinancialProblem", "SocialProblem", "PsychologicalProb", "WorkStress",
         "FamilyIncome1", "SleepingAfter", "BMI_Pregnancy", "part1_current_preg_first"]
SD_UG = ["age_numeric", "homeDistrict", "occupation", "relationshipStatus", "spouseOccupation", "polygamy", "otherChildren", "numStillBorn", "numMissCarry", "education"]
chk = []
def _regen(tag, df, label, sigcols, extra=None):
    tr, ca, te = three_way_group_split(df, label, sigcols, 0.6, 0.2, 0.2, SEED)
    S = load_split(tag, ENV)
    for k, a in [("train", tr), ("cal", ca), ("test", te)]:
        same_n = len(a) == len(S[k]); same_y = bool(same_n and np.array_equal(a[label].values, S[k][label].values))
        chk.append(dict(dataset=tag, partition=k, regenerated_n=len(a), saved_n=len(S[k]), labels_identical_in_order=same_y))
ip = lambda n: os.path.join(ENV["INTERIM"], n)
if os.path.exists(ip("pakistan_interim.csv")):
    pk = pd.read_csv(ip("pakistan_interim.csv")); _regen("pakistan", pk[SD_PK + ["y_depression"]].copy(), "y_depression", SD_PK)
if os.path.exists(ip("arab5_interim.csv")):
    ar = pd.read_csv(ip("arab5_interim.csv"))
    _regen("arab5_depression", ar[ar["dep_items_complete"] == True][SD_AR + ["y_depression"]].copy(), "y_depression", SD_AR)
    _regen("arab5_anxiety", ar[ar["anx_items_complete"] == True][SD_AR + ["y_anxiety"]].copy(), "y_anxiety", SD_AR)
if os.path.exists(ip("uganda_interim.csv")):
    ug = pd.read_csv(ip("uganda_interim.csv")); _regen("uganda", ug[SD_UG + ["y_anxiety_binary", "Anxiety", "Total Score"]].copy(), "y_anxiety_binary", SD_UG)
if not chk:
    print("WARNING: no interim files found in", ENV["INTERIM"], "- split regeneration check skipped; run notebooks 03/04 first.")
chk = pd.DataFrame(chk); save_table(chk, ENV, "S2_split_reproduction_check.csv"); print(chk.to_string())

split manifests written
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_split_reproduction_check.csv  (12 rows)
             dataset partition  regenerated_n  saved_n  labels_identical_in_order
0           pakistan     train           8402     8402                       True
1           pakistan       cal           2808     2808                       True
2           pakistan      test           2798     2798                       True
3   arab5_depression     train            902      902                       True
4   arab5_depression       cal            296      296                       True
5   arab5_depression      test            301      301                       True
6      arab5_anxiety     train            565      565                       True
7      arab5_anxiety       cal            190      190                       True
8      arab5_anxiety      test            191      191                       True
9             uganda     train           8612     8

In [6]:
cm = []
for f in sorted(glob.glob(os.path.join(ENV["ROOT"], "notebooks", "*.ipynb")) + glob.glob(os.path.join(ENV["ROOT"], "src", "*.py")) + [os.path.join(ENV["ROOT"], "config", "config.yaml")]):
    cm.append(dict(file=os.path.relpath(f, ENV["ROOT"]), bytes=os.path.getsize(f), md5=hashlib.md5(open(f, "rb").read()).hexdigest()))
save_table(pd.DataFrame(cm), ENV, "S2_code_manifest.csv")
print("\nTemplate for the Data/Code availability statement (fill in URL and commit):\n"
      "Analysis code for Study 2 (notebooks 00-28, src/ modules, config.yaml, frozen requirements, split manifests with hashed profile signatures,\n"
      "hyper-parameters and Optuna search spaces) is available at <REPOSITORY URL>, commit <HASH> (archived at <DOI>). Source datasets are available\n"
      "from the cited original publications (Pakistan, Arab-region) and https://data.mendeley.com/datasets/4swmy34scp/3 (Uganda).")

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_code_manifest.csv  (45 rows)

Template for the Data/Code availability statement (fill in URL and commit):
Analysis code for Study 2 (notebooks 00-28, src/ modules, config.yaml, frozen requirements, split manifests with hashed profile signatures,
hyper-parameters and Optuna search spaces) is available at <REPOSITORY URL>, commit <HASH> (archived at <DOI>). Source datasets are available
from the cited original publications (Pakistan, Arab-region) and https://data.mendeley.com/datasets/4swmy34scp/3 (Uganda).
